# Coin Collector: chơi thử thủ công

Chạy ô bên dưới (Shift+Enter). Nhập w (Lên), s (Xuống), a (Trái), d (Phải), q (Thoát), sau mỗi phím nhấn Enter.
Đổi bản đồ ở dòng `MAP_CHON, TEN_MAP = ...` cuối ô (MAP1, MAP2, MAP3 hoặc MAP4).

In [ ]:
# ===== CHẠY MỘT Ô NÀY LÀ CHƠI ĐƯỢC (không cần chạy các phần khác) =====
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'gymnasium', 'numpy', 'matplotlib', '-q'])

import os, json, time, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import gymnasium as gym
from gymnasium import spaces
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11


# ---- Môi trường ----

ACTION_DELTAS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
ACTION_NAMES = ['Lên', 'Xuống', 'Trái', 'Phải']
ACTION_ARROWS = ['^', 'v', '<', '>']


class CoinCollectorEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(self, config, reward_type=1):
        super().__init__()
        self.config = config
        self.rows = config["rows"]
        self.cols = config["cols"]
        self.start = tuple(config["start"])
        self.exit_pos = tuple(config["exit"])
        self.coin_dict = {tuple(k): v for k, v in config["coins"].items()}
        self.coin_pos = list(self.coin_dict.keys())
        self.n_coins = len(self.coin_pos)
        self.dis_info = {tuple(k): v for k, v in config.get("disappear_info", {}).items()}
        bp = config.get("time_bonus_pos")
        self.tb_pos = tuple(bp) if bp else None
        self.tb_val = config.get("time_bonus_value", 3) if self.tb_pos else 0
        self.max_steps = config["max_steps"]
        self.walls = {tuple(w) for w in config.get("walls", [])}
        self.rtype = reward_type
        self.n_t = self.max_steps + self.tb_val + 1
        self.n_b = 2 if self.tb_pos else 1
        self.action_space = spaces.Discrete(4)
        self.observation_space = spaces.Tuple((
            spaces.Discrete(self.rows),
            spaces.Discrete(self.cols),
            spaces.Discrete(2 ** self.n_coins),
            spaces.Discrete(self.n_t),
            spaces.Discrete(self.n_b),
        ))
        self._build_tables()
        self.reset()

    def _build_tables(self):
        R, C = self.rows, self.cols
        self._next = []
        self._valid = []
        for r in range(R):
            for c in range(C):
                row = []
                for dr, dc in ACTION_DELTAS:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < R and 0 <= nc < C and (nr, nc) not in self.walls:
                        row.append(nr * C + nc)
                    else:
                        row.append(r * C + c)
                self._next.append(row)
                self._valid.append([a for a in range(4) if row[a] != r * C + c] or [0, 1, 2, 3])
        self._coin_at = {p[0] * C + p[1]: i for i, p in enumerate(self.coin_pos)}
        self._val = [self.coin_dict[p] for p in self.coin_pos]
        self._dis = [(i, self.dis_info[p]) for i, p in enumerate(self.coin_pos) if p in self.dis_info]
        self._lim = dict(self._dis)
        self._exit_cell = self.exit_pos[0] * C + self.exit_pos[1]
        self._tb_cell = self.tb_pos[0] * C + self.tb_pos[1] if self.tb_pos else -1

    @property
    def pos(self):
        return [self.state[0], self.state[1]]

    @property
    def mask(self):
        return self.state[2]

    @property
    def time(self):
        return self.state[3]

    @property
    def bonus(self):
        return bool(self.state[4])

    @property
    def steps(self):
        return self.max_steps + self.tb_val * self.state[4] - self.state[3]

    def initial_state(self):
        return (self.start[0], self.start[1], (1 << self.n_coins) - 1, self.max_steps, 0)

    def _present(self, i):
        return bool(self.state[2] & (1 << i))

    def _coin_idx(self, p):
        p = tuple(p)
        return self.coin_pos.index(p) if p in self.coin_pos else -1

    def upper_bound(self, state, reward_type=1):
        r, c, mask, time_left, bonus = state
        steps = self.max_steps + self.tb_val * bonus - time_left
        reach = time_left + (self.tb_val if (self.tb_pos and not bonus) else 0)
        total_value, urgent_value = 0.0, 0.0
        for i, (pr, pc) in enumerate(self.coin_pos):
            if (mask >> i) & 1:
                d = abs(pr - r) + abs(pc - c)
                if d <= reach and (i not in self._lim or steps + d < self._lim[i]):
                    total_value += self._val[i]
                    if i in self._lim:
                        urgent_value += self._val[i]
        exit_ok = abs(self.exit_pos[0] - r) + abs(self.exit_pos[1] - c) <= reach
        if reward_type == 4:
            return 1.3 * total_value + (12.0 if exit_ok else 0.0)
        bound = total_value + (10.0 if exit_ok else 0.0)
        if reward_type in (3, 5):
            bound += 0.4 * urgent_value
        return bound

    def valid_actions(self, state):
        return self._valid[state[0] * self.cols + state[1]]

    def transition(self, state, action):
        r, c, mask, time_left, bonus = state
        p = self._next[r * self.cols + c][action]
        steps = self.max_steps + self.tb_val * bonus - time_left + 1
        time_left -= 1
        for i, lim in self._dis:
            if steps >= lim:
                mask &= ~(1 << i)
        gain = 0
        collected = -1
        i = self._coin_at.get(p)
        if i is not None and (mask >> i) & 1:
            gain = self._val[i]
            collected = i
            mask &= ~(1 << i)
        if p == self._tb_cell and not bonus:
            time_left += self.tb_val
            bonus = 1
        terminated = p == self._exit_cell
        truncated = time_left <= 0
        return (p // self.cols, p % self.cols, mask, time_left, bonus), gain, collected, terminated, truncated

    def _nearest_dist(self, r, c, mask):
        best = -1
        for i, (pr, pc) in enumerate(self.coin_pos):
            if (mask >> i) & 1:
                d = abs(pr - r) + abs(pc - c)
                if best < 0 or d < best:
                    best = d
        return float(best) if best >= 0 else 0.0

    def _urgency(self, r, c, mask, steps):
        u = 0.0
        for i, lim in self._dis:
            if (mask >> i) & 1:
                sl = lim - steps
                if 0 < sl <= 8:
                    pr, pc = self.coin_pos[i]
                    if abs(pr - r) + abs(pc - c) < sl:
                        u += 0.15 * self._val[i] / max(sl, 1)
        return u

    def reset(self, seed=None, options=None):
        if seed is not None:
            super().reset(seed=seed)
        self.state = self.initial_state()
        self.coins_caught = 0
        self.disappeared_caught = 0
        self.value_caught = 0
        self.reached_exit = False
        return self.state, {}

    def step(self, action):
        ns, gain, idx, terminated, truncated = self.transition(self.state, int(action))
        self.state = ns
        if idx >= 0:
            self.coins_caught += 1
            self.value_caught += gain
            if self.coin_pos[idx] in self.dis_info:
                self.disappeared_caught += 1
        if terminated:
            self.reached_exit = True
        r, c, mask, time_left, bonus = ns
        base = float(gain) + (10.0 if terminated else 0.0)
        t_ratio = max(0.0, time_left / self.max_steps)
        rt = self.rtype
        if rt == 1:
            rew = base - 0.01
        elif rt == 2:
            rew = base - 0.01 - 0.03 * self._nearest_dist(r, c, mask)
        elif rt == 3:
            rew = base - 0.05 + self._urgency(r, c, mask, self.steps)
        elif rt == 4:
            rew = gain * (1.0 + 0.3 * t_ratio) + (10.0 * (1.0 + 0.2 * t_ratio) if terminated else 0.0) - 0.02
        else:
            rew = base - 0.025 - 0.015 * self._nearest_dist(r, c, mask) + self._urgency(r, c, mask, self.steps)
        return ns, rew, terminated, truncated, {}

    def state_to_index(self, state=None):
        if state is None:
            state = self.state
        r, c, m, t, b = state
        return (((r * self.cols + c) * 2 ** self.n_coins + m) * self.n_t + t) * self.n_b + b

    def state_space_size(self):
        return self.rows * self.cols * (2 ** self.n_coins) * self.n_t * self.n_b

    def render(self):
        g = [["." for _ in range(self.cols)] for _ in range(self.rows)]
        for r, c in self.walls:
            g[r][c] = "#"
        g[self.exit_pos[0]][self.exit_pos[1]] = "E"
        if self.tb_pos and not self.bonus:
            g[self.tb_pos[0]][self.tb_pos[1]] = "T"
        for i, p in enumerate(self.coin_pos):
            if self._present(i):
                g[p[0]][p[1]] = str(self.coin_dict[p])
        g[self.state[0]][self.state[1]] = "A"
        print(f"Bước: {self.steps:3d} | Thời gian: {self.time:3d} | Coin đã thu: {self.coins_caught}/{self.n_coins}")
        print("    " + "  ".join(str(c) for c in range(self.cols)))
        for r in range(self.rows):
            print(f"{r}   " + "  ".join(g[r]))
        print()

# ---- Hàm vẽ bàn cờ và đường đi ----

COLORS = ["steelblue", "firebrick", "forestgreen", "darkorange", "purple", "saddlebrown", "teal", "crimson", "navy", "olive"]

ALGO_COLORS = {"QL": "steelblue", "MC": "firebrick", "QL-opt": "forestgreen", "MC-opt": "darkorange"}

REWARD_LABELS = {1: "R1 thưa", 2: "R2 khoảng cách", 3: "R3 khẩn cấp", 4: "R4 thời gian", 5: "R5 lai ghép"}

MAP_TITLES = {1: "Map 1 (Dễ - 4x4)", 2: "Map 2 (Trung bình - 6x6)", 3: "Map 3 (Khó - 8x8)", 4: "Map 4 (Thách thức - 7x7)"}

def cfg_parts(cfg):
    walls = {tuple(w) for w in cfg.get("walls", [])}
    cdict = {tuple(k): v for k, v in cfg["coins"].items()}
    dis = {tuple(k): v for k, v in cfg.get("disappear_info", {}).items()}
    tbp = tuple(cfg["time_bonus_pos"]) if cfg.get("time_bonus_pos") else None
    return walls, tuple(cfg["start"]), tuple(cfg["exit"]), cdict, dis, tbp, cfg.get("time_bonus_value", 3)

def draw_board(ax, cfg, scale=1.0):
    rows, cols = cfg["rows"], cfg["cols"]
    walls, start, exit_p, cdict, dis, tbp, tbv = cfg_parts(cfg)
    for r in range(rows):
        for c in range(cols):
            p = (r, c)
            if p in walls:
                fc = "#3a3a3a"
            elif p == exit_p:
                fc = "#f7c6c7"
            elif p == start:
                fc = "#d4ecd5"
            elif tbp and p == tbp:
                fc = "#d2e3f8"
            else:
                fc = "#ffffff"
            ax.add_patch(patches.Rectangle((c - 0.5, r - 0.5), 1, 1, lw=0.6, ec="#bfbfbf", fc=fc))
            if p == start:
                ax.text(c, r, "S", ha="center", va="center", fontsize=10 * scale, fontweight="bold", color="#1b4d1b")
            elif p == exit_p:
                ax.text(c, r, "E", ha="center", va="center", fontsize=10 * scale, fontweight="bold", color="#7a1c1d")
            elif tbp and p == tbp:
                ax.text(c, r, f"T+{tbv}", ha="center", va="center", fontsize=8.5 * scale, fontweight="bold", color="#1c3d7a")
            if p in cdict:
                ax.add_patch(patches.Circle((c, r), 0.28, fc="#f6c342", ec="#c99718", lw=1.0, zorder=3))
                ax.text(c, r, str(cdict[p]), ha="center", va="center", fontsize=8.5 * scale, fontweight="bold", color="#3b2b04", zorder=4)
                if p in dis:
                    ax.text(c, r + 0.38, f"<{dis[p]}", ha="center", va="center", fontsize=6.5 * scale, color="#d9383a", fontweight="bold", zorder=5)
    ax.set_xlim(-0.5, cols - 0.5)
    ax.set_ylim(rows - 0.5, -0.5)
    ax.set_xticks([])
    ax.set_yticks([])

def draw_path(ax, traj, lw=1.7, label_size=6.5):
    if len(traj) < 2:
        return
    cmap = plt.cm.viridis
    n_steps = len(traj) - 1
    pair_counts = {}
    for st in range(n_steps):
        p1, p2 = traj[st], traj[st + 1]
        color = cmap(st / max(1, n_steps - 1))
        if p1 != p2:
            ax.annotate("", xy=(p2[1], p2[0]), xytext=(p1[1], p1[0]),
                        arrowprops=dict(arrowstyle="->", color=color, lw=lw, shrinkA=4, shrinkB=4, mutation_scale=11), zorder=6)
            mx, my = (p1[1] + p2[1]) / 2.0, (p1[0] + p2[0]) / 2.0
        else:
            mx, my = float(p1[1]), float(p1[0])
        key = (min(p1, p2), max(p1, p2))
        occ = pair_counts.get(key, 0)
        pair_counts[key] = occ + 1
        off = (occ - 0.5) * 0.14 if occ > 0 else 0.0
        ax.text(mx + off, my - off, f"s{st + 1}", ha="center", va="center", fontsize=label_size, fontweight="bold",
                color="#111111", bbox=dict(boxstyle="round,pad=0.12", fc="white", ec=color, lw=0.6, alpha=0.85), zorder=7)

# ---- Bản đồ ----

MAP1 = {"rows": 4, "cols": 4, "start": (0, 0), "exit": (3, 3),
        "coins": {(0, 2): 3, (2, 0): 5}, "disappear_info": {},
        "time_bonus_pos": (1, 1), "time_bonus_value": 3, "max_steps": 9, "walls": []}

MAP2 = {"rows": 6, "cols": 6, "start": (0, 0), "exit": (5, 5),
        "coins": {(0, 3): 2, (2, 2): 4, (3, 4): 3, (5, 3): 6}, "disappear_info": {(5, 3): 10},
        "time_bonus_pos": (4, 2), "time_bonus_value": 3, "max_steps": 13, "walls": [(1, 1)]}

MAP3 = {"rows": 8, "cols": 8, "start": (0, 0), "exit": (7, 7),
        "coins": {(0, 4): 2, (2, 5): 5, (4, 1): 3, (5, 6): 7, (6, 3): 4, (7, 5): 6},
        "disappear_info": {(2, 5): 10, (5, 6): 16}, "time_bonus_pos": (5, 2), "time_bonus_value": 3,
        "max_steps": 24, "walls": [(1, 1), (1, 2), (3, 5), (4, 4)]}

MAP4 = {"rows": 7, "cols": 7, "start": (0, 0), "exit": (6, 6),
        "coins": {(3, 2): 9, (5, 5): 2, (1, 3): 4, (6, 2): 5}, "disappear_info": {(3, 2): 14, (6, 2): 20},
        "time_bonus_pos": (1, 5), "time_bonus_value": 8, "max_steps": 28,
        "walls": [(2, 1), (3, 1), (4, 1), (4, 2), (2, 2), (0, 5), (5, 3)]}

# ---- Hàm chơi ----

ACTION_KEYS = {"w": 0, "s": 1, "a": 2, "d": 3}
DIVIDER = "=" * 25


def print_game_header(cfg, map_name):
    print(DIVIDER)
    print(f"        CHẾ ĐỘ TỰ CHƠI COIN COLLECTOR: {map_name.upper()}")
    print(DIVIDER)
    print(f"Kích thước bản đồ : {cfg['rows']} hàng x {cfg['cols']} cột")
    print(f"Giới hạn thời gian: {cfg['max_steps']} bước đi")
    print(f"Điểm xuất phát (S): {tuple(cfg['start'])} | Điểm thoát hiểm (E): {tuple(cfg['exit'])}")
    walls = cfg.get("walls", [])
    print(f"Vị trí tường (#)  : {walls if walls else 'Không có'}")
    if cfg.get("time_bonus_pos"):
        print(f"Ô Time Bonus (T)  : {tuple(cfg['time_bonus_pos'])} (+{cfg.get('time_bonus_value', 3)} bước, dùng 1 lần)")
    else:
        print("Ô Time Bonus (T)  : Không có trên bản đồ này")
    print()
    print("DANH SÁCH CÁC ĐỒNG XU:")
    dis = cfg.get("disappear_info", {})
    for idx, (pos, val) in enumerate(cfg["coins"].items(), 1):
        status = f"Biến mất sau {dis[pos]} bước (<{dis[pos]})" if pos in dis else "Cố định suốt màn chơi"
        print(f"  Coin {idx}: Tọa độ {pos} | Giá trị: {val:2d} điểm | Trạng thái: {status}")
    print()
    print("CHÚ GIẢI: A=Agent  S=Start  E=Exit  T=Time Bonus  #=Tường  [số]=Đồng xu")
    print("ĐIỀU KHIỂN:  w = Lên  |  s = Xuống  |  a = Trái  |  d = Phải  |  q = Thoát")
    print(DIVIDER)


def draw_manual_result(config, map_name, path, score, mission, coins, n_coins, exited, quit_early):
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    draw_board(ax, config, 1.2)
    draw_path(ax, path, lw=2, label_size=8)
    steps = len(path) - 1
    if quit_early:
        status = "thoát giữa chừng"
    elif exited:
        status = "đã tới Exit"
    else:
        status = "hết thời gian"
    ax.set_title(f"{map_name}: đường đi của người chơi ({status})\nđiểm nhiệm vụ {mission:.0f}, coin {coins}/{n_coins}, {steps} bước, tổng reward {score:+.3f}", fontsize=10)
    plt.tight_layout()
    plt.show()


def play_manual(config, map_name="Bản đồ", reward_type=1, seed=42, scripted=None):
    print_game_header(config, map_name)
    interactive = scripted is None
    if interactive:
        input("Nhấn ENTER để bắt đầu chơi...")
    env = CoinCollectorEnv(config, reward_type=reward_type)
    env.reset(seed=seed)
    score, step, history = 0.0, 0, []
    path = [tuple(env.pos)]
    quit_early = False
    queue = list(scripted) if scripted is not None else None
    while True:
        print(DIVIDER)
        print(f" MÀN CHƠI: {map_name} | BƯỚC: {step:2d}/{env.max_steps + env.tb_val * int(env.bonus)} | THỜI GIAN CÒN: {env.time:2d}")
        env.render()
        print(f"Vị trí: {tuple(env.pos)} | Điểm tích lũy: {score:+.3f} | Coin đã thu: {env.coins_caught}/{env.n_coins}")
        if interactive:
            raw = ""
            while raw not in ACTION_KEYS:
              raw = input("Nhập hành động (w/s/a/d/q): ").strip().lower()
              if raw == "q":
                    quit_early = True
                    break
            if quit_early:
                print("Bạn đã thoát màn chơi.")
                break
        else:
            if not queue:
                break
            raw = queue.pop(0)
        action = ACTION_KEYS[raw]
        prev = tuple(env.pos)
        was_bonus = bool(env.bonus)
        _, rew, ter, trun, _ = env.step(action)
        step += 1
        score += rew
        cur = tuple(env.pos)
        path.append(cur)
        desc = f"Bước {step}: đi {ACTION_NAMES[action]} đến {cur}"
        if cur == prev:
            desc += " -> va chạm biên/tường, đứng yên"
        elif cur in config["coins"] and not env._present(env._coin_idx(cur)):
            desc += f" -> thu được coin {config['coins'][cur]} điểm"
        elif config.get("time_bonus_pos") and cur == tuple(config["time_bonus_pos"]) and env.bonus and not was_bonus:
            desc += f" -> kích hoạt Time Bonus (+{config.get('time_bonus_value')} bước)"
        history.append((desc, rew))
        if ter or trun:
            break
    mission = env.value_caught + (10 if env.reached_exit else 0)
    print(DIVIDER)
    if quit_early:
        print("KẾT QUẢ: Thoát giữa chừng")
    else:
        print("KẾT QUẢ: Đã đến Exit" if env.reached_exit else "KẾT QUẢ: Hết thời gian hoặc dừng sớm")
    env.render()
    print(f"Tổng số bước: {step} | Coin thu được: {env.coins_caught}/{env.n_coins} | Điểm nhiệm vụ: {mission} | Tổng reward: {score:.3f}")
    print("LỊCH SỬ CÁC BƯỚC ĐI:")
    for desc, rew in history:
        print(f"  {desc} (reward: {rew:+.2f})")
    print(DIVIDER)
    draw_manual_result(config, map_name, path, score, mission, env.coins_caught, env.n_coins, env.reached_exit, quit_early)

# ---- Chọn bản đồ và chơi ----
# Đổi MAP_CHON thành MAP1, MAP2, MAP3 hoặc MAP4; TEN_MAP là tên hiển thị.
MAP_CHON, TEN_MAP = MAP2, "Map 2 (Trung bình - 6x6)"
play_manual(MAP_CHON, map_name=TEN_MAP, reward_type=3)
